# broadcast-source-fanout — faded example 1: Fill the source rank's send loop in a manual broadcast

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-source-fanout`. The last cell reports your progress on the `Generative: Broadcast source fan-out` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Broadcast source fan-out` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-source-fanout`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-source-fanout"
DD_SUBTOPIC = "Generative: Broadcast source fan-out"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A broadcast copies the source rank's tensor to all other ranks: the source loops `dist.send` to each destination while receivers `dist.recv` then `copy_` into their local buffer. We emulate this with per-rank Python buffers so the fan-out logic is explicit. The key step is the source-side loop that pushes an independent copy to every non-source rank.

## Faded exercise 1

Complete `manual_broadcast(payload, src, world_size)` so it returns a `(world_size, *payload.shape)` tensor where every rank's buffer equals the source payload but is an *independent* allocation (no aliasing). The per-rank buffer setup and the final stack are given; you must fill in the source fan-out loop that copies the payload into every non-source rank's buffer.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
from torch import Tensor

def manual_broadcast(payload: Tensor, src: int, world_size: int):
    buffers = {r: t.zeros_like(payload) for r in range(world_size)}
    buffers[src] = payload.clone()
    # TODO: fill in this step — read the prompt cell above
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    return t.stack([buffers[r] for r in range(world_size)], dim=0)


def _test():
    import torch as t
    t.manual_seed(0)
    payload = t.randn(3)
    out = manual_broadcast(payload, src=2, world_size=4)
    assert tuple(out.shape) == (4, 3)
    # every rank equals source
    assert t.allclose(out, payload.expand(4, 3))
    # independence: stacked rows are distinct allocations from source
    out[0, 0] += 100.0
    assert not t.allclose(out[0], out[1]), "buffers alias each other"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
from torch import Tensor

def manual_broadcast(payload: Tensor, src: int, world_size: int):
    buffers = {r: t.zeros_like(payload) for r in range(world_size)}
    buffers[src] = payload.clone()
    for r in range(world_size):
        if r == src:
            continue
        buffers[r].copy_(buffers[src].clone())
    return t.stack([buffers[r] for r in range(world_size)], dim=0)
```
</details>